#### 그래프 스테이트와 노드 함수 정의

In [1]:
import operator
from typing import Annotated, Any
from langchain_core.messages import SystemMessage, HumanMessage, BaseMessage
from langchain_openai import ChatOpenAI
from pydantic import BaseModel, Field
from dotenv import load_dotenv
load_dotenv("../.env")

True

In [2]:
# 그래프의 스테이트 정의
class State(BaseModel):
    query: str
    messages: Annotated[list[BaseMessage], operator.add] = Field(default=[])

In [3]:
# 메시지를 추가하는 노드 함수
def add_message(state: State) -> dict[str, Any]:
    additional_messages = []
    if not state.messages:
        additional_messages.append(SystemMessage(content="당신은 최소한의 응답을 하는 대화 에이전트입니다."))
    additional_messages.append(HumanMessage(content=state.query))
    return {"messages": additional_messages}

In [4]:
# LLM 응답을 추가하는 노드 함수
def llm_response(state: State) -> dict[str, Any]:
    llm = ChatOpenAI(model="gpt-6-luna")
    ai_message = llm.invoke(state.messages)
    return {"messages": [ai_message]}

#### 체크포인트 내용을 표시하는 함수 정의

In [5]:
from pprint import pprint
from langchain_core.runnables import RunnableConfig
from langgraph.checkpoint.base import BaseCheckpointSaver

def print_checkpoint_dump(checkpointer: BaseCheckpointSaver, config: RunnableConfig):
    checkpoint_tuple = checkpointer.get_tuple(config)
    print("체크포인트 데이터:")
    pprint(checkpoint_tuple.checkpoint)
    print("\n메타데이터:")
    pprint(checkpoint_tuple.metadata)

#### 그래프 정의 및 컴파일

In [6]:
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import InMemorySaver

In [7]:
# 그래프 설정
graph = StateGraph(State)
graph.add_node("add_message", add_message)
graph.add_node("llm_response", llm_response)
graph.set_entry_point("add_message")
graph.add_edge(
    # start_key: str | list[str],
    "add_message",
    # end_key: str
    "llm_response")
graph.add_edge(
    # start_key: str | list[str],
    "llm_response",
    # end_key: str
    END)

In [8]:
# 체크포인터 설정
checkpointer = InMemorySaver()

In [9]:
# 그래프 컴파일
compiled_graph = graph.compile(
    # checkpointer: Checkpointer = None,
    checkpointer=checkpointer
    # *,
    # cache: BaseCache[Unknown] | None = None,
    # store: BaseStore | None = None,
    # interrupt_before: list[str] | Literal['*'] | None = None,
    # interrupt_after: list[str] | Literal['*'] | None = None,
    # debug: bool = False,
    # name: str | None = None,
    # transformers: Sequence[(tuple[str, ...]) -> Any] | None = None
    )

#### 실행해 작동 확인하기

In [10]:
config = {"configurable": {"thread_id": "example-1"}}
user_query = State(query="제가 좋아하는 것은 찹살떡입니다. 기억해 주세요.")
first_response = compiled_graph.invoke(user_query, config)
first_response

c:\Users\limux\Documents\1. Projects\llm-engineer-roadmap\02-rag-agents-langchain-langgraph\.venv\Lib\site-packages\langsmith\_internal\_addressing.py:104: LangSmithBetaWarning: Addressing runs with `langsmith.address` is in beta and is enabled per workspace. A workspace without it rejects the run, so the trace is lost rather than falling back to a project. The behavior may change without notice.
  _warn_once(


{'query': '제가 좋아하는 것은 찹살떡입니다. 기억해 주세요.',
 'messages': [SystemMessage(content='당신은 최소한의 응답을 하는 대화 에이전트입니다.', additional_kwargs={}, response_metadata={}),
  HumanMessage(content='제가 좋아하는 것은 찹살떡입니다. 기억해 주세요.', additional_kwargs={}, response_metadata={}),
  AIMessage(content='기억해 둘게요. 좋아하는 음식은 찹쌀떡이군요!', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 91, 'prompt_tokens': 43, 'total_tokens': 134, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 61, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': 0, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-6-luna', 'system_fingerprint': None, 'id': 'chatcmpl-EkkOEcCN0gVcXP44wrCtCf5KcZm5M', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a1e305-02e4-727e-99d1-910573bbb44d-0', tool_cal

In [11]:
# 이제 체크포인터에 대해 list 함수를 사용해 체크포인트 정보가 어떻게 변화했는지 살펴보자
for checkpoint in checkpointer.list(config):
    print(checkpoint)

Deserializing unregistered type __main__.State from checkpoint. This will be blocked in a future version. Set LANGGRAPH_STRICT_MSGPACK=true to block now, or add to allowed_msgpack_modules to allow explicitly: [('__main__', 'State')]


CheckpointTuple(config={'configurable': {'thread_id': 'example-1', 'checkpoint_ns': '', 'checkpoint_id': '1f1e1860-8bcd-6fd9-8002-f1377cd3c644'}}, checkpoint={'v': 4, 'ts': '2026-11-16T06:22:01.365909+00:00', 'id': '1f1e1860-8bcd-6fd9-8002-f1377cd3c644', 'channel_versions': {'__start__': '00000000000000000000000000000002.0.9021692126584525', 'query': '00000000000000000000000000000002.0.9021692126584525', 'messages': '00000000000000000000000000000004.0.24431557764540612', 'branch:to:add_message': '00000000000000000000000000000003.0.2755348154708118', 'branch:to:llm_response': '00000000000000000000000000000004.0.24431557764540612'}, 'versions_seen': {'__input__': {}, '__start__': {'__start__': '00000000000000000000000000000001.0.2961768912893348'}, 'add_message': {'branch:to:add_message': '00000000000000000000000000000002.0.9021692126584525'}, 'llm_response': {'branch:to:llm_response': '00000000000000000000000000000003.0.2755348154708118'}}, 'updated_channels': ['messages'], 'channel_val

In [12]:
# 다음 코드를 실행해 최신 체크포인트의 상세 데이터를 가져와 본다.
print_checkpoint_dump(checkpointer, config)

체크포인트 데이터:
{'channel_values': {'messages': [SystemMessage(content='당신은 최소한의 응답을 하는 대화 에이전트입니다.', additional_kwargs={}, response_metadata={}),
                                 HumanMessage(content='제가 좋아하는 것은 찹살떡입니다. 기억해 주세요.', additional_kwargs={}, response_metadata={}),
                                 AIMessage(content='기억해 둘게요. 좋아하는 음식은 찹쌀떡이군요!', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 91, 'prompt_tokens': 43, 'total_tokens': 134, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 61, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': 0, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-6-luna', 'system_fingerprint': None, 'id': 'chatcmpl-EkkOEcCN0gVcXP44wrCtCf5KcZm5M', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run

In [14]:
# 다시 그래프를 실행해 본다
# invoke 함수의 인자에는 과거 대화 기억을 테스트하는 질문을 설정한다.
# 그래프의 스테이트가 저장돼 있고 과거 기억을 추적할 수 있다면 이 질문에 답할 수 있을 것
user_query = State(query="제가 좋아하는 것이 뭔지 기억하세요?")
second_response = compiled_graph.invoke(user_query, config)
second_response

{'query': '제가 좋아하는 것이 뭔지 기억하세요?',
 'messages': [SystemMessage(content='당신은 최소한의 응답을 하는 대화 에이전트입니다.', additional_kwargs={}, response_metadata={}),
  HumanMessage(content='제가 좋아하는 것은 찹살떡입니다. 기억해 주세요.', additional_kwargs={}, response_metadata={}),
  AIMessage(content='기억해 둘게요. 좋아하는 음식은 찹쌀떡이군요!', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 91, 'prompt_tokens': 43, 'total_tokens': 134, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 61, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': 0, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-6-luna', 'system_fingerprint': None, 'id': 'chatcmpl-EkkOEcCN0gVcXP44wrCtCf5KcZm5M', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a1e305-02e4-727e-99d1-910573bbb44d-0', tool_calls=[], 

In [15]:
# 다시 list 함수를 실행하고 Checkpoint의 channel_values 내용을 확인해 보겠다.
for checkpoint in checkpointer.list(config):
    print(checkpoint)

CheckpointTuple(config={'configurable': {'thread_id': 'example-1', 'checkpoint_ns': '', 'checkpoint_id': '1f1e18dd-c1fd-6101-8006-ad512e1ce21a'}}, checkpoint={'v': 4, 'ts': '2026-11-16T07:18:02.490700+00:00', 'id': '1f1e18dd-c1fd-6101-8006-ad512e1ce21a', 'channel_versions': {'__start__': '00000000000000000000000000000006.0.2669337590715346', 'query': '00000000000000000000000000000006.0.2669337590715346', 'messages': '00000000000000000000000000000008.0.33572910427477887', 'branch:to:add_message': '00000000000000000000000000000007.0.028997702708036477', 'branch:to:llm_response': '00000000000000000000000000000008.0.33572910427477887'}, 'versions_seen': {'__input__': {}, '__start__': {'__start__': '00000000000000000000000000000005.0.18625360355601006'}, 'add_message': {'branch:to:add_message': '00000000000000000000000000000006.0.2669337590715346'}, 'llm_response': {'branch:to:llm_response': '00000000000000000000000000000007.0.028997702708036477'}}, 'updated_channels': ['messages'], 'channe

In [16]:
print_checkpoint_dump(checkpointer, config)

체크포인트 데이터:
{'channel_values': {'messages': [SystemMessage(content='당신은 최소한의 응답을 하는 대화 에이전트입니다.', additional_kwargs={}, response_metadata={}),
                                 HumanMessage(content='제가 좋아하는 것은 찹살떡입니다. 기억해 주세요.', additional_kwargs={}, response_metadata={}),
                                 AIMessage(content='기억해 둘게요. 좋아하는 음식은 찹쌀떡이군요!', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 91, 'prompt_tokens': 43, 'total_tokens': 134, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 61, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': 0, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-6-luna', 'system_fingerprint': None, 'id': 'chatcmpl-EkkOEcCN0gVcXP44wrCtCf5KcZm5M', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run

In [18]:
# 마지막으로 thread_id를 변경해 실행한 경우를 확인해 보겠다.
config = {"configurable": {"thread_id": "example-2"}}
user_query = State(query="제가 좋아하는 것은 뭔가요?")
other_thread_response = compiled_graph.invoke(user_query, config)
pprint(other_thread_response)

{'messages': [SystemMessage(content='당신은 최소한의 응답을 하는 대화 에이전트입니다.', additional_kwargs={}, response_metadata={}),
              HumanMessage(content='제가 좋아하는 것은 뭔가요?', additional_kwargs={}, response_metadata={}),
              AIMessage(content='아직 말씀해 주신 게 없어서 모르겠어요.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 51, 'prompt_tokens': 37, 'total_tokens': 88, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 29, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': 0, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-6-luna', 'system_fingerprint': None, 'id': 'chatcmpl-EklSK15JJjJcd3YpZH1lDwrcjyTVn', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a1e343-8831-73e1-8c5b-ce4c429ca106-0', tool_calls=[], invalid_tool_calls=[], usa

In [ ]:
for checkpoint in checkpointer.list(config):
    pprint(checkpoint)